# Build Your Own Vector Search (MongoDB Atlas + Voyage AI)

This notebook walks through the full pipeline **manually**, step by step, so
you understand what's actually happening before you see it fully automated
in Part 2 (the Mastra Studio agent demo):

1. Explore the schema of the shared `streaming_catalog.titles` collection
2. Copy a small subset into your own personal sandbox collection
3. Embed the `synopsis` field yourself using the **Voyage AI** embeddings API
4. Create your own self-managed Atlas **Vector Search** index on those embeddings
5. Run a `$vectorSearch` query against your own self-managed index
6. Drop that index, then build a second index on the same documents using
   Atlas's **Automated Embedding** (`autoEmbed`) — notice there's no embedding
   code at all this time
7. Query the `autoEmbed` index with plain text instead of a pre-computed vector
8. Combine semantic search with a `release_year` metadata filter
9. Insert a brand-new title with zero embedding code, and watch it show up in
   semantic search live — the moment that really sells `autoEmbed`
10. Compare both approaches side by side
11. Clean up your sandbox (so the shared cluster doesn't accumulate search
    indexes across every attendee)

**Prerequisites:** you've already run `cp .env.example .env` and filled in
`WORKSHOP_USER_ID`, `MONGODB_DEMO_CONNECTION_STRING`, and
`MONGODB_ADMIN_READONLY_CONNECTION_STRING` for the main Node app — this
notebook reuses that same `.env` file, no extra credentials needed.

```bash
pip install -r notebooks/requirements.txt
jupyter lab notebooks/01_build_vector_search.ipynb
```


## Before you start

Run this in your terminal from the repo root (not in the notebook), then
launch Jupyter from the same activated environment so it picks up the
right kernel and dependencies automatically:

```bash
python3 -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r notebooks/requirements.txt
jupyter lab notebooks/01_build_vector_search.ipynb
```

**Prerequisite:** Python 3.10+ and pip. If the next cell fails with
`ModuleNotFoundError`, you likely skipped this step or forgot to activate
`.venv` before launching Jupyter.

In [ ]:
import os
import time
import json
from datetime import datetime, timezone

from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
import voyageai
import certifi

load_dotenv()

WORKSHOP_USER_ID = os.environ["WORKSHOP_USER_ID"]
DEMO_CONNECTION_STRING = os.environ["MONGODB_DEMO_CONNECTION_STRING"]
ADMIN_READONLY_CONNECTION_STRING = os.environ["MONGODB_ADMIN_READONLY_CONNECTION_STRING"]

print(f"Workshop user: {WORKSHOP_USER_ID}")


## 0. Check the workshop is active and fetch your Voyage AI key

Every attendee's Voyage AI key is admin-controlled, exactly like the Grove
AI Gateway key used by the agent in Part 2. It lives in the same
`workshop_admin.settings` document and is fetched here read-only — you never
need your own personal Voyage account or key.


In [ ]:
admin_client = MongoClient(
    ADMIN_READONLY_CONNECTION_STRING,
    serverSelectionTimeoutMS=8000,
    tlsCAFile=certifi.where(),
)
settings = admin_client["workshop_admin"]["settings"].find_one({"_id": "workshop"})

if settings is None:
    raise RuntimeError("Workshop settings have not been provisioned yet. Ask your admin.")

now = datetime.now(timezone.utc)
starts_at = settings["startsAt"]
expires_at = settings["expiresAt"]
if starts_at.tzinfo is None:
    starts_at = starts_at.replace(tzinfo=timezone.utc)
if expires_at.tzinfo is None:
    expires_at = expires_at.replace(tzinfo=timezone.utc)

if not settings.get("enabled", False):
    raise RuntimeError("The workshop has been disabled by the admin.")
if now < starts_at:
    raise RuntimeError(f"The workshop hasn't started yet (starts at {starts_at.isoformat()}).")
if now > expires_at:
    raise RuntimeError(f"The workshop window has ended (expired at {expires_at.isoformat()}). Thanks for attending!")

VOYAGE_API_KEY = settings.get("voyageApiKey")
if not VOYAGE_API_KEY:
    raise RuntimeError("Voyage API key is not configured yet. Ask your admin.")

print("Workshop is active. Voyage API key fetched.")


## 1. Explore the schema

Before building anything, look at what's actually in the shared catalog
collection every attendee reads from and writes to.


In [ ]:
demo_client = MongoClient(
    DEMO_CONNECTION_STRING,
    serverSelectionTimeoutMS=8000,
    tlsCAFile=certifi.where(),
)
catalog_db = demo_client["streaming_catalog"]
titles = catalog_db["titles"]

sample_doc = titles.find_one({}, {"_id": 0})
print("Example document fields:\n")
for key, value in sample_doc.items():
    preview = repr(value)
    if len(preview) > 100:
        preview = preview[:100] + "..."
    print(f"  {key:<18} {type(value).__name__:<8} {preview}")

print(f"\nTotal documents in streaming_catalog.titles: {titles.count_documents({})}")


### Peek at the production vector index (Automated Embedding)

This is the index your admin already built for you — it embeds `synopsis`
automatically, server-side, using the `voyage-4` model (`autoEmbed` field
type). You're about to build your own version of this by hand.


In [ ]:
print("Existing search indexes on streaming_catalog.titles:\n")
for index in titles.list_search_indexes():
    print(json.dumps(index, indent=2, default=str))


## 2. Copy a working subset into your own sandbox collection

To avoid touching the shared production collection/index while you
experiment, copy a handful of documents into a personal sandbox collection
named after your `WORKSHOP_USER_ID`.


In [ ]:
SANDBOX_COLLECTION_NAME = f"titles_sandbox_{WORKSHOP_USER_ID}"
sandbox = catalog_db[SANDBOX_COLLECTION_NAME]

# Start fresh in case you're re-running this notebook
sandbox.drop()

subset = list(titles.find({}, {"_id": 0}).limit(8))
sandbox.insert_many(subset)
print(f"Copied {len(subset)} documents into {catalog_db.name}.{SANDBOX_COLLECTION_NAME}")


## 3. Embed the `synopsis` field with Voyage AI

This is the step Atlas's Automated Embedding does for you automatically in
production. Here, you call the Voyage AI embeddings API directly so you can
see exactly what an "embedding" is: a list of floating-point numbers.


In [ ]:
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

docs = list(sandbox.find({}))
synopses = [doc["synopsis"] for doc in docs]

result = vo.embed(synopses, model="voyage-4", input_type="document")
print(f"Embedded {len(result.embeddings)} documents, "
      f"{len(result.embeddings[0])} dimensions each, "
      f"{result.total_tokens} tokens used.")

print("\nFirst few values of the first embedding:")
print(result.embeddings[0][:8], "...")

for doc, embedding in zip(docs, result.embeddings):
    sandbox.update_one({"_id": doc["_id"]}, {"$set": {"plot_embedding": embedding}})

print("\nWrote `plot_embedding` back onto every sandbox document.")


## 4. Create your own vector search index

Now that your documents have embeddings stored as a `plot_embedding` field,
build a **self-managed** vector search index on top of them — the classic
pattern, as opposed to the `autoEmbed` index you saw in step 1.


In [ ]:
INDEX_NAME = f"plot_vector_index_{WORKSHOP_USER_ID}"

search_index_model = SearchIndexModel(
    name=INDEX_NAME,
    type="vectorSearch",
    definition={
        "fields": [
            {"type": "vector", "path": "plot_embedding", "numDimensions": 1024, "similarity": "cosine"},
            {"type": "filter", "path": "genres"},
            {"type": "filter", "path": "release_year"},
        ]
    },
)

sandbox.create_search_index(model=search_index_model)
print(f"Submitted creation of '{INDEX_NAME}'. Waiting for it to become queryable...")

while True:
    indexes = list(sandbox.list_search_indexes(INDEX_NAME))
    if indexes and indexes[0].get("queryable"):
        print("Index is queryable! Full definition:\n")
        print(json.dumps(indexes[0], indent=2, default=str))
        break
    time.sleep(5)


## 5. Search it! Try one of the workshop's example prompts

Embed a query string yourself (`input_type="query"`), then run a
`$vectorSearch` aggregation against your own index — the same core operation
the agent runs for you automatically in Part 2.


In [ ]:
query_text = "a feel-good documentary about food"

query_embedding = vo.embed([query_text], model="voyage-4", input_type="query").embeddings[0]

results = sandbox.aggregate([
    {
        "$vectorSearch": {
            "index": INDEX_NAME,
            "path": "plot_embedding",
            "queryVector": query_embedding,
            "numCandidates": 50,
            "limit": 5,
        }
    },
    {
        "$project": {
            "_id": 0,
            "title": 1,
            "genres": 1,
            "synopsis": 1,
            "score": {"$meta": "vectorSearchScore"},
        }
    },
])

for doc in results:
    print(f"{doc['score']:.4f}  {doc['title']}  {doc['genres']}")
    print(f"           {doc['synopsis']}\n")


## 6. Switch to Automated Embedding: clean up the self-managed index first

Atlas Search index counts are limited per cluster, especially on shared
tiers — with several attendees running this notebook at once, build only
one sandbox index at a time. Drop the self-managed index you just used, and
also remove the `plot_embedding` field you wrote by hand, so the collection
starts clean and nothing left over can influence or fake out the `autoEmbed`
showcase coming up next.


In [ ]:
sandbox.drop_search_index(INDEX_NAME)
print(f"Dropped self-managed search index '{INDEX_NAME}'.")

sandbox.update_many({}, {"$unset": {"plot_embedding": ""}})
print("Removed the 'plot_embedding' field from every sandbox document — starting clean before building the autoEmbed index.")


## 7. Build your own Automated Embedding (`autoEmbed`) index

Same sandbox collection, same documents — but this time you won't call the
Voyage AI embeddings API yourself at all. The index definition points
directly at the `synopsis` text field, and Atlas computes and stores the
embeddings for you, server-side, using the same `voyage-4` model.


In [ ]:
AUTOEMBED_INDEX_NAME = f"plot_autoembed_index_{WORKSHOP_USER_ID}"

autoembed_index_model = SearchIndexModel(
    name=AUTOEMBED_INDEX_NAME,
    type="vectorSearch",
    definition={
        "fields": [
            {"type": "autoEmbed", "path": "synopsis", "model": "voyage-4", "modality": "text"},
            {"type": "filter", "path": "genres"},
            {"type": "filter", "path": "release_year"},
        ]
    },
)

sandbox.create_search_index(model=autoembed_index_model)
print(f"Submitted creation of '{AUTOEMBED_INDEX_NAME}'. Waiting for it to become queryable...")

while True:
    indexes = list(sandbox.list_search_indexes(AUTOEMBED_INDEX_NAME))
    if indexes and indexes[0].get("queryable"):
        print("Index is queryable! Full definition:\n")
        print(json.dumps(indexes[0], indent=2, default=str))
        break
    time.sleep(5)


## 8. Query it with plain text — no pre-computed vector needed

This time, pass your query as plain text instead of calling `vo.embed()`
yourself first. Atlas embeds the query text automatically, server-side, the
same way it already embedded every document above.


In [ ]:
results = sandbox.aggregate([
    {
        "$vectorSearch": {
            "index": AUTOEMBED_INDEX_NAME,
            "path": "synopsis",
            "query": {"text": query_text},
            "numCandidates": 50,
            "limit": 5,
        }
    },
    {
        "$project": {
            "_id": 0,
            "title": 1,
            "genres": 1,
            "synopsis": 1,
            "score": {"$meta": "vectorSearchScore"},
        }
    },
])

for doc in results:
    print(f"{doc['score']:.4f}  {doc['title']}  {doc['genres']}")
    print(f"           {doc['synopsis']}\n")


## 9. Combine semantic search with a metadata filter (`release_year`)

`$vectorSearch`'s `filter` option works the same way regardless of whether
the index uses a self-managed vector or `autoEmbed` — it operates on a
separately indexed filter field, independent of how the embedding itself
was computed. Narrow your search to only recent titles.


In [ ]:
results = sandbox.aggregate([
    {
        "$vectorSearch": {
            "index": AUTOEMBED_INDEX_NAME,
            "path": "synopsis",
            "query": {"text": query_text},
            "filter": {"release_year": {"$gte": 2022}},
            "numCandidates": 50,
            "limit": 5,
        }
    },
    {
        "$project": {
            "_id": 0,
            "title": 1,
            "genres": 1,
            "release_year": 1,
            "synopsis": 1,
            "score": {"$meta": "vectorSearchScore"},
        }
    },
])

print("Only titles from 2022 onward:\n")
for doc in results:
    print(f"{doc['score']:.4f}  {doc['title']} ({doc['release_year']})  {doc['genres']}")
    print(f"           {doc['synopsis']}\n")


## 10. The real win: insert a new title with zero embedding code

This is the moment that `autoEmbed` is really for. Insert a brand-new title
document the normal way — a plain `insert_one()`, no embedding call anywhere
— then immediately search for it. Atlas embeds new/updated documents
automatically in the background (via change streams), the same mechanism
that powers attendees' live-inserted titles in Part 2.

Run the next two cells in order. Sync latency is normally small, but if your
new title isn't in the results yet, just re-run the search cell again after
a few seconds.


In [ ]:
new_title = {
    "title": "Grandma's Kitchen Confidential",
    "type": "movie",
    "genres": ["documentary", "comedy"],
    "synopsis": (
        "A retired line cook turned grandmother invites a rotating cast of "
        "chaotic home bakers into her kitchen for one heartwarming, hilarious "
        "cook-off after another."
    ),
    "release_year": 2024,
    "ownerId": WORKSHOP_USER_ID,
}

sandbox.insert_one(new_title)
print(f"Inserted '{new_title['title']}' into {catalog_db.name}.{SANDBOX_COLLECTION_NAME} — just a plain insert, no embedding code.")


In [ ]:
results = sandbox.aggregate([
    {
        "$vectorSearch": {
            "index": AUTOEMBED_INDEX_NAME,
            "path": "synopsis",
            "query": {"text": "a heartwarming cooking show about grandmothers baking together"},
            "numCandidates": 50,
            "limit": 5,
        }
    },
    {
        "$project": {
            "_id": 0,
            "title": 1,
            "genres": 1,
            "synopsis": 1,
            "score": {"$meta": "vectorSearchScore"},
        }
    },
])

print("If \"Grandma's Kitchen Confidential\" isn't listed below yet, Atlas is still embedding it in the background — re-run this cell after a few seconds.\n")
for doc in results:
    print(f"{doc['score']:.4f}  {doc['title']}  {doc['genres']}")
    print(f"           {doc['synopsis']}\n")


## 11. Compare: what you just did vs. Atlas Automated Embedding

| Step | What you just did manually | What `autoEmbed` does instead |
|---|---|---|
| Embedding documents | Called `vo.embed(..., input_type="document")` yourself and wrote `plot_embedding` by hand | Atlas calls Voyage AI for you at insert/update time, server-side — no embedding field or API call in your app code |
| Embedding queries | Called `vo.embed(..., input_type="query")` yourself before searching | Atlas embeds your `$vectorSearch` query text automatically — you just pass `{"text": "..."}` instead of a vector |
| Index definition | `type: "vector"` field, pointing at your own stored `plot_embedding` field | `type: "autoEmbed"` field, pointing at the *source text* field (`synopsis`) directly — no embedding field needed at all |
| Keeping embeddings fresh on new inserts | You'd have to re-embed and re-write manually every time | Atlas re-embeds automatically via change streams — this is exactly what powers attendees' live-inserted titles in Part 2 |

Both approaches produce the same kind of Atlas Vector Search index under the
hood and both support metadata filtering (`genres`, `release_year`, etc.) —
the difference is entirely about *who* computes the embeddings and *when*.
Self-managed embeddings (what you just did) give you full control over the
embedding model/version and let you embed with any provider; Automated
Embedding trades that control for zero embedding-pipeline code to
write or maintain.


## 12. Clean up your sandbox

Atlas Search index counts are limited per cluster, especially on
lower/shared tiers — with several attendees running this notebook, it's
important to clean up your own index and sandbox collection once you're
done exploring.


In [ ]:
sandbox.drop_search_index(AUTOEMBED_INDEX_NAME)
print(f"Dropped search index '{AUTOEMBED_INDEX_NAME}'. It may take a few seconds to fully disappear from list_search_indexes().")

sandbox.drop()
print(f"Dropped sandbox collection '{SANDBOX_COLLECTION_NAME}'.")


## 13. Next: try the fully automated version in Mastra Studio

You've now built the vector search pipeline by hand: schema → embed →
index → search. Part 2 of the workshop shows the same capability wrapped in
a natural-language agent, backed by Atlas's Automated Embedding on the
shared production collection:

```bash
npm install
npm run dev
```

Try the exact same prompt you just ran manually:

> "I want a feel-good documentary about food."

...and watch the tool-call trace in Mastra Studio — same `$vectorSearch`
operation, zero embedding code in the agent itself. See
[`demo/workshop_script.md`](../demo/workshop_script.md) for the full live
demo flow.
